# Day 1: Classical ML Baseline (TF-IDF + Logistic Regression)
**Config Source:** `.env` + `configs/app_config.yaml` (`engine/settings.py`) — Zero hardcoded URLs in notebook cells.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
ACTIVE_DOMAIN = SETTINGS.active_domain  # Override here or via BOOTCAMP_DOMAIN in .env

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: Establish the Deterministic ML Floor First**
Trains `TF-IDF + LogisticRegression` on the active domain's 10 incidents to benchmark sub-5ms severity classification before introducing vector DBs or SLMs.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

dbg.start_run("day01_classical")
with dbg.start_span(f"Day1_ClassicalBaseline_{ACTIVE_INCIDENT['incident_id']}", span_kind="CHAIN") as root_span:
    t0 = time.perf_counter()
    vec = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
    X = vec.fit_transform([inc["raw_log"] for inc in INCIDENTS])
    y = [inc["severity"] for inc in INCIDENTS]
    clf = LogisticRegression(max_iter=500).fit(X, y)

    pred = clf.predict(vec.transform([ACTIVE_INCIDENT["raw_log"]]))[0]
    probs = dict(zip(clf.classes_, [round(float(p), 3) for p in clf.predict_proba(vec.transform([ACTIVE_INCIDENT["raw_log"]]))[0]]))
    lat_ms = round((time.perf_counter() - t0) * 1000, 2)
    dbg.record_step("tfidf_logreg_predict", ACTIVE_INCIDENT["raw_log"], {"predicted": pred, "probs": probs}, lat_ms, span_kind="CHAIN")

viz.render_story(
    title=f"Day 1 Classical ML Floor — {ACTIVE_INCIDENT['incident_id']}",
    stages=[
        {"name": "TF-IDF Vectorization + Logistic Regression", "badge": "CLASSICAL ML", "latency_ms": lat_ms, "summary": f"Predicted Severity: {pred} (Expected: {ACTIVE_INCIDENT['severity']})", "scores": [{"label": k, "score": v} for k, v in probs.items()], "payload": probs}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Vertex AI AutoML / Gemini comparison enabled.")
else:
    print("ENABLE_VERTEX_AI=false -> Classical baseline executed 100% locally.")


### Cell 6 — Automated Validation Gate


In [ ]:
assert pred in ("P1", "P2", "P3")
print("[PASS] Day 1 Classical Baseline Gate passed!")
